In [ ]:
# ==========================================
# NLGCL Kaggle 运行指南 (All-in-One)
# ==========================================
# 本 Notebook 用于在 Kaggle 环境中配置并运行 NLGCL 模型。
# 
# ## 前置条件
# 1. **GPU 环境**: 请确保 Notebook 设置中 Accelerator 选择 GPU (如 GPU P100 或 T4)。
# 2. **数据集**: 请上传本地 `dataset` 文件夹到 Kaggle Datasets (命名推荐 `nlgcl-dataset`)，并挂载到 Notebook 中。
#    - Kaggle 挂载路径通常为 `/kaggle/input/nlgcl-dataset`。
#
# ------------------------------------------
# STEP 1: 环境检测与代码准备
# ------------------------------------------
import os

# 定义你的 GitHub 仓库地址 (请替换为你的实际地址)
GITHUB_REPO_URL = "https://github.com/Jinfeng-Xu/NLGCL.git" 
REPO_NAME = "NLGCL" # 仓库名

if os.path.exists('/kaggle/working'):
    # 在 Kaggle 环境下
    print("Detected Kaggle environment.")
    
    # 克隆代码 (如果不存在)
    if not os.path.exists(REPO_NAME):
        cmd = f"git clone {GITHUB_REPO_URL} {REPO_NAME}"
        print(f"Executing: {cmd}")
        os.system(cmd)
    
    # 进入代码目录
    if os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
        print(f"Changed directory to {os.getcwd()}")
else:
    print("Not in Kaggle environment, assuming local run.")
    # 如果在本地，不需要 clone，直接运行即可

# ------------------------------------------
# STEP 2: 安装依赖 (极速版 - 使用稳定版 PyTorch)
# ------------------------------------------
print("Installing dependencies...")

# 1. 解决基础依赖冲突
!pip install "colorama>=0.4.6"

# 2. 安装 RecBole
!pip install --upgrade recbole

# 3. 关键步骤：重装 PyTorch 为 2.5.1 稳定版
# Kaggle 默认的 PyTorch (2.9+) 太新，没有 PyG 的二进制包，导致需要耗时编译。
# 我们直接安装兼容性最好的 2.5.1 版本，这样可以秒装 PyG。
print("Switching to stable PyTorch 2.5.1 to enable fast binary installation...")

# 卸载当前版本 (可选，防止冲突)
# !pip uninstall -y torch torchvision torchaudio

# 安装 PyTorch 2.5.1
!pip install torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --extra-index-url https://download.pytorch.org/whl/cu124

# 4. 安装 PyG (使用官方 Wheel，无需编译)
print("Installing PyG binaries...")
wheel_url = "https://data.pyg.org/whl/torch-2.5.1+cu124.html"
!pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_url}
!pip install torch_geometric

# 5. 安装其他 loose 依赖
!pip install -r requirements.txt

import torch
print(f"Final PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# ------------------------------------------
# STEP 3: 数据集准备 (自动寻找与链接)
# ------------------------------------------
import shutil

# 自动寻找 Kaggle Input 中的 dataset 目录
def find_dataset_root(search_path='/kaggle/input'):
    for root, dirs, files in os.walk(search_path):
        if 'yelp.inter' in files:
            if os.path.basename(root) == 'yelp':
                return os.path.dirname(root)
            return os.path.dirname(root)
    return None

if os.path.exists('/kaggle/working'):
    current_dataset_link = 'dataset'
    
    # 1. 寻找挂载的数据集
    found_path = find_dataset_root()
    if not found_path:
        # 尝试默认路径
        potential_paths = [
            "/kaggle/input/nlgcl-dataset",
            "/kaggle/input/nlgcl-dataset/dataset",
            "/kaggle/input/dataset"
        ]
        for p in potential_paths:
            if os.path.exists(os.path.join(p, 'yelp', 'yelp.inter')):
                found_path = p
                break
    
    if found_path:
        print(f"Found dataset at: {found_path}")
        
        # 2. 清理旧的 dataset 目录/链接
        if os.path.exists(current_dataset_link):
            if os.path.islink(current_dataset_link):
                os.unlink(current_dataset_link)
            else:
                shutil.rmtree(current_dataset_link)
        
        # 3. 复制数据集
        print("Copying dataset to working directory (required for write access)...")
        shutil.copytree(found_path, current_dataset_link)
        print("Dataset copied successfully.")
        
    else:
        print("Error: Could not find 'yelp/yelp.inter' in /kaggle/input.")
        !find /kaggle/input -maxdepth 3
else:
    print("Local environment, skipping dataset setup.")

!ls -l dataset/yelp/

# ------------------------------------------
# STEP 4: 强制写入过滤配置 (覆盖 yelp.yaml)
# ------------------------------------------
yelp_config_content = """
load_col:
  inter: [user_id, item_id, rating]
ITEM_ID_FIELD: item_id
RATING_FIELD: rating

# 核心过滤配置：过滤交互少于 15 次的用户和物品
user_inter_num_interval: "[15,inf)"
item_inter_num_interval: "[15,inf)"

val_interval:
  rating: "[3,inf)"

warm_up_step: 40

# ----------------
# 训练与日志配置
# ----------------
epochs: 300
train_batch_size: 4096
learner: adam
learning_rate: 0.001
eval_step: 1
stopping_step: 10

# 评估指标
metrics: ["Recall", "NDCG"]
topk: [10, 20, 50]
valid_metric: NDCG@20

# 日志输出控制
verbose: True
state: INFO
"""

os.makedirs('properties', exist_ok=True)
with open('properties/yelp.yaml', 'w') as f:
    f.write(yelp_config_content)
print("Updated properties/yelp.yaml with filtering rules.")

# ------------------------------------------
# STEP 5: 运行训练
# ------------------------------------------
if os.path.exists("saved"):
    import glob
    for pth in glob.glob("saved/*.pth"):
        os.remove(pth)

print("Starting training...")
# 确保 recbole 可用 (有时重装 torch 后需要重新 verify)
try:
    import recbole
except ImportError:
    !pip install recbole

!python main.py --dataset=yelp